# 1 Extracting Data

### 1.0 Imports

In [3]:
import pandas as pd
from fitparse import FitFile
from collections import Counter
from pathlib import Path
from datetime import datetime, timedelta
import numpy as np

In [4]:
# ============================================================
# Parameters
# ============================================================
first_date =  "2026-06-03"

LOCAL_OFFSET = pd.Timedelta(hours=2)

GARMIN_EPOCH = datetime(1989, 12, 31)

COGNITIVE_LOGS = {
    "2026-06-03": [
        ("09:14", "09:59"),
        ("11:00", "11:24"),
        ("11:43", "12:36"),
        ("13:53", "15:11"),
        ("16:17", "17:20"),
    ],
    "2026-06-04": [
        ("10:26", "11:12"),
        ("11:22", "11:30"),
        ("11:38", "12:50"),
        ("14:04", "15:24"),
        ("16:35", "16:57"),
        ("17:25", "17:45"),
        ("17:59", "18:34"),
    ],
}

In [5]:
# ============================================================
# Select date and inspect WELLNESS files
# ============================================================
data_dir = Path(f"../data/{first_date}")
wellness_files = sorted(data_dir.glob("*_WELLNESS.fit"))

print(f"Found {len(wellness_files)} WELLNESS files\n")

summary = []

for file in wellness_files:
    fitfile = FitFile(str(file))

    stress_count = 0

    for _ in fitfile.get_messages("stress_level"):
        stress_count += 1

    summary.append({
        "file": file.name,
        "stress_records": stress_count
    })

summary_df = pd.DataFrame(summary)

display(summary_df)

Found 7 WELLNESS files



,file,stress_records
0,444927859217_WELLNESS.fit,477
1,444930358648_WELLNESS.fit,5
2,444931351296_WELLNESS.fit,4
3,444984229414_WELLNESS.fit,219
4,445043193231_WELLNESS.fit,231
5,445097413482_WELLNESS.fit,229
6,445149205969_WELLNESS.fit,196


### 1.1 Stress

In [7]:
def extract_stress(
    wellness_files,
    curr_date,
    local_offset
):
    """
    Extract stress data from all WELLNESS files for one day.
    """

    all_rows = []

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("stress_level"):

            row = msg.get_values()

            all_rows.append({
                "source_file": file.name,
                "timestamp": row["stress_level_time"] + local_offset,
                "stress": row["stress_level_value"],
                "body_battery": row.get("body_battery")
            })

    stress_df = pd.DataFrame(all_rows)

    # Garmin missing value
    stress_df["stress"] = (
        stress_df["stress"]
        .replace(-1, pd.NA)
    )

    stress_df = (
        stress_df
        .sort_values("timestamp")
        .drop_duplicates(
            subset="timestamp",
            keep="last"
        )
        .reset_index(drop=True)
    )

    stress_df = stress_df[
        stress_df["timestamp"].dt.date
        == pd.to_datetime(curr_date).date()
    ].copy()

    return stress_df

In [8]:
stress_df = extract_stress(
    wellness_files,
    first_date,
    LOCAL_OFFSET
)

print("Rows:", len(stress_df))
print("Start:", stress_df["timestamp"].min())
print("End:", stress_df["timestamp"].max())

display(stress_df.head())
display(stress_df.tail())

Rows: 1360
Start: 2026-06-03 00:01:00
End: 2026-06-03 23:59:00


,source_file,timestamp,stress,body_battery
0,444927859217_WELLNESS.fit,2026-06-03 00:01:00,12,None
1,444927859217_WELLNESS.fit,2026-06-03 00:02:00,20,None
2,444927859217_WELLNESS.fit,2026-06-03 00:04:00,22,None
3,444927859217_WELLNESS.fit,2026-06-03 00:05:00,39,None
4,444927859217_WELLNESS.fit,2026-06-03 00:07:00,16,None


,source_file,timestamp,stress,body_battery
1355,445149205969_WELLNESS.fit,2026-06-03 23:55:00,17,None
1356,445149205969_WELLNESS.fit,2026-06-03 23:56:00,15,None
1357,445149205969_WELLNESS.fit,2026-06-03 23:57:00,13,None
1358,445149205969_WELLNESS.fit,2026-06-03 23:58:00,17,None
1359,445149205969_WELLNESS.fit,2026-06-03 23:59:00,24,None


### 1.2 Monitor Data

In [10]:
def extract_monitoring(
    wellness_files,
    curr_date
):
    """
    Extract monitoring data from all WELLNESS files for one day.
    Includes heart rate and activity-related monitoring fields.
    """

    rows = []

    target_date = pd.to_datetime(curr_date).date()

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("monitoring"):

            row = msg.get_values()

            if "timestamp_16" not in row:
                continue

            timestamp = (
                datetime.combine(
                    target_date,
                    datetime.min.time()
                )
                + timedelta(seconds=row["timestamp_16"])
            )

            rows.append({
                "source_file": file.name,
                "timestamp": timestamp,

                "heart_rate": row.get("heart_rate"),

                "cycles": row.get("cycles"),
                "active_calories": row.get("active_calories"),
                "active_time": row.get("active_time")
            })

    monitoring_df = (
        pd.DataFrame(rows)
        .sort_values("timestamp")
        .drop_duplicates(subset="timestamp", keep="last")
        .reset_index(drop=True)
    )

    return monitoring_df

In [11]:
monitoring_df = extract_monitoring(
    wellness_files,
    first_date
)

print("Rows:", len(monitoring_df))
print("Start:", monitoring_df["timestamp"].min())
print("End:", monitoring_df["timestamp"].max())

display(monitoring_df.head())
display(monitoring_df.tail())

Rows: 1304
Start: 2026-06-03 00:00:56
End: 2026-06-03 18:12:12


,source_file,timestamp,heart_rate,cycles,active_calories,active_time
0,445097413482_WELLNESS.fit,2026-06-03 00:00:56,72.0,NaN,NaN,NaN
1,445097413482_WELLNESS.fit,2026-06-03 00:01:56,71.0,NaN,NaN,NaN
2,445097413482_WELLNESS.fit,2026-06-03 00:02:56,74.0,NaN,NaN,NaN
3,445097413482_WELLNESS.fit,2026-06-03 00:03:56,73.0,NaN,NaN,NaN
4,445097413482_WELLNESS.fit,2026-06-03 00:04:56,80.0,NaN,NaN,NaN


,source_file,timestamp,heart_rate,cycles,active_calories,active_time
1299,445097413482_WELLNESS.fit,2026-06-03 18:07:12,68.0,NaN,NaN,NaN
1300,445097413482_WELLNESS.fit,2026-06-03 18:09:12,76.0,NaN,NaN,NaN
1301,445097413482_WELLNESS.fit,2026-06-03 18:10:12,73.0,NaN,NaN,NaN
1302,445097413482_WELLNESS.fit,2026-06-03 18:11:12,72.0,NaN,NaN,NaN
1303,445097413482_WELLNESS.fit,2026-06-03 18:12:12,70.0,NaN,NaN,NaN


In [12]:
monitoring_df.notna().sum().sort_values(ascending=False)

source_file        1304
timestamp          1304
heart_rate         1161
cycles               86
active_calories      86
active_time          86
dtype: int64

### 1.3 Body Battery

In [14]:
def extract_body_battery(
    wellness_files,
    curr_date,
    local_offset
):
    """
    Extract body battery data from all WELLNESS files for one day.
    """

    all_rows = []

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("stress_level"):

            row = msg.get_values()

            all_rows.append({
                "source_file": file.name,
                "timestamp": row["stress_level_time"] + local_offset,
                "body_battery": row["unknown_3"]
            })

    body_battery_df = pd.DataFrame(all_rows)

    body_battery_df["body_battery"] = (
        body_battery_df["body_battery"]
        .replace(-1, pd.NA)
    )

    body_battery_df = (
        body_battery_df
        .sort_values("timestamp")
        .drop_duplicates(
            subset="timestamp",
            keep="last"
        )
        .reset_index(drop=True)
    )

    body_battery_df = body_battery_df[
        body_battery_df["timestamp"].dt.date
        == pd.to_datetime(curr_date).date()
    ].copy()

    return body_battery_df

In [15]:
body_battery_df = extract_body_battery(
    wellness_files,
    first_date,
    LOCAL_OFFSET
)

print("Rows:", len(body_battery_df))
print("Start:", body_battery_df["timestamp"].min())
print("End:", body_battery_df["timestamp"].max())

display(body_battery_df.head())
display(body_battery_df.tail())

Rows: 1360
Start: 2026-06-03 00:01:00
End: 2026-06-03 23:59:00


,source_file,timestamp,body_battery
0,444927859217_WELLNESS.fit,2026-06-03 00:01:00,32
1,444927859217_WELLNESS.fit,2026-06-03 00:02:00,32
2,444927859217_WELLNESS.fit,2026-06-03 00:04:00,32
3,444927859217_WELLNESS.fit,2026-06-03 00:05:00,32
4,444927859217_WELLNESS.fit,2026-06-03 00:07:00,32


,source_file,timestamp,body_battery
1355,445149205969_WELLNESS.fit,2026-06-03 23:55:00,21
1356,445149205969_WELLNESS.fit,2026-06-03 23:56:00,21
1357,445149205969_WELLNESS.fit,2026-06-03 23:57:00,23
1358,445149205969_WELLNESS.fit,2026-06-03 23:58:00,23
1359,445149205969_WELLNESS.fit,2026-06-03 23:59:00,23


### 1.4 Respirational Rate

In [17]:
def extract_respiration(
    wellness_files,
    curr_date,
    local_offset,
    garmin_epoch
):
    """
    Extract respiration data from all WELLNESS files for one day.
    """

    all_rows = []

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("unknown_297"):

            row = msg.get_values()

            all_rows.append({
                "source_file": file.name,
                "timestamp": (
                    garmin_epoch
                    + timedelta(seconds=row["unknown_253"])
                    + local_offset
                ),
                "respiration_rate": row["unknown_0"] / 100
            })

    respiration_df = pd.DataFrame(all_rows)

    respiration_df.loc[
        respiration_df["respiration_rate"] < 0,
        "respiration_rate"
    ] = pd.NA

    respiration_df = (
        respiration_df
        .sort_values("timestamp")
        .drop_duplicates(
            subset="timestamp",
            keep="last"
        )
        .reset_index(drop=True)
    )

    respiration_df = respiration_df[
        respiration_df["timestamp"].dt.date
        == pd.to_datetime(curr_date).date()
    ].copy()

    return respiration_df

In [18]:
respiration_df = extract_respiration(
    wellness_files,
    first_date,
    LOCAL_OFFSET,
    GARMIN_EPOCH
)

print("Rows:", len(respiration_df))
print("Start:", respiration_df["timestamp"].min())
print("End:", respiration_df["timestamp"].max())

display(respiration_df.head())
display(respiration_df.tail())

Rows: 1439
Start: 2026-06-03 00:01:00
End: 2026-06-03 23:59:00


,source_file,timestamp,respiration_rate
0,444927859217_WELLNESS.fit,2026-06-03 00:01:00,13.58
1,444927859217_WELLNESS.fit,2026-06-03 00:02:00,15.36
2,444927859217_WELLNESS.fit,2026-06-03 00:03:00,14.91
3,444927859217_WELLNESS.fit,2026-06-03 00:04:00,16.58
4,444927859217_WELLNESS.fit,2026-06-03 00:05:00,NaN


,source_file,timestamp,respiration_rate
1434,445149205969_WELLNESS.fit,2026-06-03 23:55:00,14.91
1435,445149205969_WELLNESS.fit,2026-06-03 23:56:00,15.25
1436,445149205969_WELLNESS.fit,2026-06-03 23:57:00,15.00
1437,445149205969_WELLNESS.fit,2026-06-03 23:58:00,14.33
1438,445149205969_WELLNESS.fit,2026-06-03 23:59:00,15.16


### 1.5 Event Logs - Activity Type

In [20]:
ACTIVITY_MAP = {
    0: "generic",
    1: "running",
    2: "cycling",
    6: "walking",
    8: "sedentary"
}

In [21]:
def extract_activity_events(
    wellness_files,
    GARMIN_EPOCH,
    LOCAL_OFFSET
):
    """
    Extract Garmin auto-detected activity events.

    Decodes:
        unknown_14 -> activity code
        unknown_15 -> start timestamp

    Returns:
        timestamp
        start_timestamp
        end_timestamp
        activity_code
        activity_type
        duration_min
    """

    ACTIVITY_MAP = {
        0: "generic",
        1: "running",
        2: "cycling",
        6: "walking",
        8: "sedentary"
    }

    rows = []

    for file in wellness_files:

        fitfile = FitFile(str(file))

        for msg in fitfile.get_messages("event"):

            row = msg.get_values()

            # Skip non-auto-activity markers
            if row.get("event") != 54:
                continue

            activity_code = row.get("unknown_14")
            duration_min = row.get("data")
            start_seconds = row.get("unknown_15")

            if (
                activity_code is None
                or duration_min is None
                or start_seconds is None
            ):
                continue

            start_timestamp = (
                GARMIN_EPOCH
                + timedelta(seconds=int(start_seconds))
                + LOCAL_OFFSET
            )

            end_timestamp = (
                start_timestamp
                + timedelta(minutes=int(duration_min))
            )

            rows.append({
                "source_file": file.name,
                "timestamp": row.get("timestamp"),

                "activity_code": activity_code,
                "activity_type": ACTIVITY_MAP.get(
                    activity_code,
                    "unknown"
                ),

                "duration_min": int(duration_min),

                "start_timestamp": start_timestamp,
                "end_timestamp": end_timestamp
            })

    events_df = (
        pd.DataFrame(rows)
        .sort_values("start_timestamp")
        .reset_index(drop=True)
    )

    return events_df

In [22]:
events_df = extract_activity_events(
    wellness_files,
    GARMIN_EPOCH,
    LOCAL_OFFSET
)

print("Rows:", len(events_df))

if len(events_df) > 0:
    print("Start:", events_df["start_timestamp"].min())
    print("End:", events_df["end_timestamp"].max())

display(events_df.head())
display(events_df.tail())

print(events_df["activity_type"].value_counts())

Rows: 240
Start: 2026-06-02 23:20:00
End: 2026-06-03 23:31:00


,source_file,timestamp,activity_code,activity_type,duration_min,start_timestamp,end_timestamp
0,444927859217_WELLNESS.fit,2026-06-02 22:18:00,8,sedentary,43,2026-06-02 23:20:00,2026-06-03 00:03:00
1,444927859217_WELLNESS.fit,2026-06-02 22:19:00,0,generic,1,2026-06-03 00:03:00,2026-06-03 00:04:00
2,444927859217_WELLNESS.fit,2026-06-02 22:58:00,8,sedentary,39,2026-06-03 00:04:00,2026-06-03 00:43:00
3,444927859217_WELLNESS.fit,2026-06-02 22:59:00,0,generic,1,2026-06-03 00:43:00,2026-06-03 00:44:00
4,444927859217_WELLNESS.fit,2026-06-02 23:06:00,8,sedentary,7,2026-06-03 00:44:00,2026-06-03 00:51:00


,source_file,timestamp,activity_code,activity_type,duration_min,start_timestamp,end_timestamp
235,445149205969_WELLNESS.fit,2026-06-03 21:12:00,0,generic,10,2026-06-03 22:47:00,2026-06-03 22:57:00
236,445149205969_WELLNESS.fit,2026-06-03 21:13:00,8,sedentary,1,2026-06-03 22:57:00,2026-06-03 22:58:00
237,445149205969_WELLNESS.fit,2026-06-03 21:42:00,0,generic,29,2026-06-03 22:58:00,2026-06-03 23:27:00
238,445149205969_WELLNESS.fit,2026-06-03 21:44:00,8,sedentary,2,2026-06-03 23:27:00,2026-06-03 23:29:00
239,445149205969_WELLNESS.fit,2026-06-03 21:46:00,0,generic,2,2026-06-03 23:29:00,2026-06-03 23:31:00


activity_type
generic      115
sedentary    110
walking        8
unknown        4
cycling        2
running        1
Name: count, dtype: int64


### 1.6 Merging

In [24]:
def build_daily_df(
    stress_df,
    body_battery_df,
    respiration_df,
    monitoring_df,
    events_df,
    curr_date
):
    """
    Merge all extracted Garmin data into a unified
    1-minute dataframe for one day.

    Activity labels come from events_df
    (Garmin auto activity detection), not monitoring_df.
    """

    # ========================================================
    # Ensure datetime dtype
    # ========================================================

    for df in [
        stress_df,
        body_battery_df,
        respiration_df,
        monitoring_df,
        events_df
    ]:
        if len(df) > 0:
            for col in df.columns:
                if "timestamp" in col:
                    df[col] = pd.to_datetime(df[col])

    # ========================================================
    # Keep only target day
    # ========================================================

    target_date = pd.to_datetime(curr_date).date()

    stress_df = stress_df[
        stress_df["timestamp"].dt.date == target_date
    ].copy()

    body_battery_df = body_battery_df[
        body_battery_df["timestamp"].dt.date == target_date
    ].copy()

    respiration_df = respiration_df[
        respiration_df["timestamp"].dt.date == target_date
    ].copy()

    monitoring_df = monitoring_df[
        monitoring_df["timestamp"].dt.date == target_date
    ].copy()

    # ========================================================
    # Full minute timeline
    # ========================================================

    full_index = pd.date_range(
        start=pd.Timestamp(curr_date),
        periods=1440,
        freq="1min"
    )

    daily_df = pd.DataFrame(index=full_index)

    # ========================================================
    # Helper
    # ========================================================

    def prepare_minute_df(df, value_col):

        temp = df.copy()

        temp["timestamp"] = temp["timestamp"].dt.floor("min")

        return (
            temp.groupby("timestamp")[value_col]
            .first()
            .to_frame()
        )

    # ========================================================
    # Physiological features
    # ========================================================

    stress_min = prepare_minute_df(
        stress_df,
        "stress"
    )

    body_battery_min = prepare_minute_df(
        body_battery_df,
        "body_battery"
    )

    respiration_min = prepare_minute_df(
        respiration_df,
        "respiration_rate"
    )

    heart_rate_min = prepare_minute_df(
        monitoring_df,
        "heart_rate"
    )


    # ========================================================
    # Merge physiological features
    # ========================================================

    for feature_df in [
        stress_min,
        body_battery_min,
        respiration_min,
        heart_rate_min,
    ]:
        daily_df = daily_df.join(feature_df)

    # ========================================================
    # Activity labels from event log
    # ========================================================

    daily_df["activity_code"] = np.nan
    daily_df["activity_type"] = np.nan

    events_day = events_df[
        events_df["start_timestamp"].dt.date == target_date
    ].copy()

    for _, event in events_day.iterrows():

        mask = (
            (daily_df.index >= event["start_timestamp"])
            &
            (daily_df.index < event["end_timestamp"])
        )

        daily_df.loc[
            mask,
            "activity_code"
        ] = event["activity_code"]

        daily_df.loc[
            mask,
            "activity_type"
        ] = event["activity_type"]

    # ========================================================
    # Finalize
    # ========================================================

    daily_df.index.name = "timestamp"

    return daily_df.reset_index()

In [25]:
daily_df = build_daily_df(
    stress_df,
    body_battery_df,
    respiration_df,
    monitoring_df,
    events_df,
    first_date
)

print("Shape:", daily_df.shape)

print("\nMissing values:")
print(daily_df.isna().sum())

print("\nDate range:")
print(daily_df["timestamp"].min())
print(daily_df["timestamp"].max())

display(daily_df.head(20))

Shape: (1440, 7)

Missing values:
timestamp             0
stress              221
body_battery         80
respiration_rate    307
heart_rate          498
activity_code        32
activity_type        32
dtype: int64

Date range:
2026-06-03 00:00:00
2026-06-03 23:59:00


C:\Users\gerib\AppData\Local\Temp\ipykernel_2428\391049552.py:144: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'generic' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  daily_df.loc[


,timestamp,stress,body_battery,respiration_rate,heart_rate,activity_code,activity_type
0,2026-06-03 00:00:00,NaN,NaN,NaN,72.0,NaN,NaN
1,2026-06-03 00:01:00,12,32.0,13.58,71.0,NaN,NaN
2,2026-06-03 00:02:00,20,32.0,15.36,74.0,NaN,NaN
3,2026-06-03 00:03:00,NaN,NaN,14.91,73.0,0.0,generic
4,2026-06-03 00:04:00,22,32.0,16.58,80.0,8.0,sedentary
5,2026-06-03 00:05:00,39,32.0,NaN,76.0,8.0,sedentary
6,2026-06-03 00:06:00,NaN,NaN,15.25,69.0,8.0,sedentary
7,2026-06-03 00:07:00,16,32.0,13.75,71.0,8.0,sedentary
8,2026-06-03 00:08:00,18,32.0,15.08,72.0,8.0,sedentary
9,2026-06-03 00:09:00,19,32.0,14.75,NaN,8.0,sedentary


### 1.7 Cognitive Target

In [27]:
def add_cognitive_target(
    daily_df,
    curr_date,
    cognitive_logs
):
    """
    Add binary cognitive target.

    is_cognitive = 1 during logged cognitive sessions
    is_cognitive = 0 otherwise
    """

    daily_df = daily_df.copy()

    daily_df["timestamp"] = pd.to_datetime(
        daily_df["timestamp"]
    )

    daily_df["is_cognitive"] = 0

    sessions = cognitive_logs.get(
        curr_date,
        []
    )

    for start_str, end_str in sessions:

        start_dt = pd.Timestamp(
            f"{curr_date} {start_str}"
        )

        end_dt = pd.Timestamp(
            f"{curr_date} {end_str}"
        )

        mask = (
            (daily_df["timestamp"] >= start_dt)
            &
            (daily_df["timestamp"] < end_dt)
        )

        daily_df.loc[
            mask,
            "is_cognitive"
        ] = 1

    return daily_df

In [28]:
daily_df = add_cognitive_target(
    daily_df,
    "2026-06-03",
    COGNITIVE_LOGS
)

print(
    daily_df["is_cognitive"]
    .value_counts()
)

is_cognitive
0    1177
1     263
Name: count, dtype: int64


### 1.8 Save Function

In [30]:
def save_full_df(
    full_df,
    output_dir="../final_dfs",
    filename="full_df.csv"
):
    """
    Save the full extracted dataframe as one CSV file.
    """

    output_dir = Path(output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    output_file = output_dir / filename

    full_df.to_csv(
        output_file,
        index=False
    )

    print(f"Saved: {output_file}")

### 1.9 Data Extraction

In [32]:
def extract_day(
    curr_date,
    local_offset,
    garmin_epoch,
    cognitive_logs
):
    """
    Complete extraction pipeline for one day.
    Returns one 1-minute dataframe for curr_date.
    """

    data_dir = Path(f"../data/{curr_date}")

    wellness_files = sorted(
        data_dir.glob("*_WELLNESS.fit")
    )

    stress_df = extract_stress(
        wellness_files,
        curr_date,
        local_offset
    )

    body_battery_df = extract_body_battery(
        wellness_files,
        curr_date,
        local_offset
    )

    respiration_df = extract_respiration(
        wellness_files,
        curr_date,
        local_offset,
        garmin_epoch
    )

    monitoring_df = extract_monitoring(
        wellness_files,
        curr_date
    )

    events_df = extract_activity_events(
        wellness_files,
        garmin_epoch,
        local_offset
    )

    daily_df = build_daily_df(
        stress_df,
        body_battery_df,
        respiration_df,
        monitoring_df,
        events_df,
        curr_date
    )

    daily_df = add_cognitive_target(
        daily_df,
        curr_date,
        cognitive_logs
    )

    #daily_df["date"] = pd.to_datetime(curr_date).date()

    return daily_df

### 1.10 Extracting multiple days at once

In [34]:
# ============================================================
# Process all dates into one full dataframe
# ============================================================

dates = [
    "2026-06-03",
    "2026-06-04"
]

daily_dfs = []

for curr_date in dates:

    print(f"\nProcessing {curr_date}")

    day_df = extract_day(
        curr_date,
        LOCAL_OFFSET,
        GARMIN_EPOCH,
        COGNITIVE_LOGS
    )

    daily_dfs.append(day_df)

    print("Shape:", day_df.shape)

    print("\nMissing values:")
    print(day_df.isna().sum())

    display(day_df.head())

full_df = (
    pd.concat(
        daily_dfs,
        ignore_index=True
    )
)

save_full_df(
    full_df,
    filename="full_df.csv"
)

print("\nFull dataframe shape:", full_df.shape)

print("\nFull dataframe missing values:")
print(full_df.isna().sum())

display(full_df.head())
display(full_df.tail())


Processing 2026-06-03
Shape: (1440, 8)

Missing values:
timestamp             0
stress              221
body_battery         80
respiration_rate    307
heart_rate          498
activity_code        32
activity_type        32
is_cognitive          0
dtype: int64


C:\Users\gerib\AppData\Local\Temp\ipykernel_2428\391049552.py:144: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'generic' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  daily_df.loc[


,timestamp,stress,body_battery,respiration_rate,heart_rate,activity_code,activity_type,is_cognitive
0,2026-06-03 00:00:00,NaN,NaN,NaN,72.0,NaN,NaN,0
1,2026-06-03 00:01:00,12,32.0,13.58,71.0,NaN,NaN,0
2,2026-06-03 00:02:00,20,32.0,15.36,74.0,NaN,NaN,0
3,2026-06-03 00:03:00,NaN,NaN,14.91,73.0,0.0,generic,0
4,2026-06-03 00:04:00,22,32.0,16.58,80.0,8.0,sedentary,0



Processing 2026-06-04
Shape: (1440, 8)

Missing values:
timestamp             0
stress              144
body_battery         54
respiration_rate    273
heart_rate          490
activity_code        77
activity_type        77
is_cognitive          0
dtype: int64


C:\Users\gerib\AppData\Local\Temp\ipykernel_2428\391049552.py:144: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'generic' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  daily_df.loc[


,timestamp,stress,body_battery,respiration_rate,heart_rate,activity_code,activity_type,is_cognitive
0,2026-06-04 00:00:00,NaN,NaN,NaN,68.0,NaN,NaN,0
1,2026-06-04 00:01:00,NaN,NaN,15.00,77.0,NaN,NaN,0
2,2026-06-04 00:02:00,17,23.0,13.66,72.0,NaN,NaN,0
3,2026-06-04 00:03:00,20,23.0,14.16,70.0,NaN,NaN,0
4,2026-06-04 00:04:00,NaN,NaN,15.41,65.0,NaN,NaN,0


Saved: ..\final_dfs\full_df.csv

Full dataframe shape: (2880, 8)

Full dataframe missing values:
timestamp             0
stress              365
body_battery        134
respiration_rate    580
heart_rate          988
activity_code       109
activity_type       109
is_cognitive          0
dtype: int64


,timestamp,stress,body_battery,respiration_rate,heart_rate,activity_code,activity_type,is_cognitive
0,2026-06-03 00:00:00,NaN,NaN,NaN,72.0,NaN,NaN,0
1,2026-06-03 00:01:00,12,32.0,13.58,71.0,NaN,NaN,0
2,2026-06-03 00:02:00,20,32.0,15.36,74.0,NaN,NaN,0
3,2026-06-03 00:03:00,NaN,NaN,14.91,73.0,0.0,generic,0
4,2026-06-03 00:04:00,22,32.0,16.58,80.0,8.0,sedentary,0


,timestamp,stress,body_battery,respiration_rate,heart_rate,activity_code,activity_type,is_cognitive
2875,2026-06-04 23:55:00,15,26.0,15.41,NaN,NaN,NaN,0
2876,2026-06-04 23:56:00,15,26.0,15.33,NaN,NaN,NaN,0
2877,2026-06-04 23:57:00,14,26.0,15.08,NaN,NaN,NaN,0
2878,2026-06-04 23:58:00,14,26.0,14.58,NaN,NaN,NaN,0
2879,2026-06-04 23:59:00,16,26.0,14.41,NaN,NaN,NaN,0


# 2 EDA

In [36]:
# Check one date
check_date = "2026-06-04"

check_df = full_df[
    full_df["timestamp"].dt.date == pd.to_datetime(check_date).date()
]

display(check_df.head())
print(check_df.isna().sum())

,timestamp,stress,body_battery,respiration_rate,heart_rate,activity_code,activity_type,is_cognitive
1440,2026-06-04 00:00:00,NaN,NaN,NaN,68.0,NaN,NaN,0
1441,2026-06-04 00:01:00,NaN,NaN,15.00,77.0,NaN,NaN,0
1442,2026-06-04 00:02:00,17,23.0,13.66,72.0,NaN,NaN,0
1443,2026-06-04 00:03:00,20,23.0,14.16,70.0,NaN,NaN,0
1444,2026-06-04 00:04:00,NaN,NaN,15.41,65.0,NaN,NaN,0


timestamp             0
stress              144
body_battery         54
respiration_rate    273
heart_rate          490
activity_code        77
activity_type        77
is_cognitive          0
dtype: int64


In [37]:
# Target Distribution
check_df["is_cognitive"].value_counts()

is_cognitive
0    1157
1     283
Name: count, dtype: int64

# 3 Preprocessing

In [39]:
daily_df_processed = daily_df.copy()

### 3.1 Missing Values